# Dynamo frontend smoke test

This notebook calls the OpenAI-compatible Dynamo HTTP frontend. Start the frontend on port 8000 (for example `python -m dynamo.frontend --http-port 8000` with workers registered), then run the cells in order.

Endpoints covered: `GET /health`, `GET /v1/models`, `POST /v1/chat/completions`.

In [1]:
import json
import urllib.error
import urllib.request

BASE_URL = "http://localhost:8000"
TIMEOUT_SEC = 120


from typing import Any, Dict, Optional, Tuple


def _request(method: str, path: str, body: Optional[Dict[str, Any]] = None) -> Tuple[int, bytes]:
    url = BASE_URL.rstrip("/") + path
    data = json.dumps(body).encode("utf-8") if body is not None else None
    req = urllib.request.Request(
        url,
        data=data,
        method=method,
        headers={"Content-Type": "application/json"} if body is not None else {},
    )
    with urllib.request.urlopen(req, timeout=TIMEOUT_SEC) as resp:
        return resp.status, resp.read()


def get_json(path: str) -> Tuple[int, dict]:
    status, raw = _request("GET", path, None)
    return status, json.loads(raw.decode("utf-8"))


def post_json(path: str, payload: dict) -> Tuple[int, dict]:
    status, raw = _request("POST", path, payload)
    return status, json.loads(raw.decode("utf-8"))


print(f"Target: {BASE_URL}")

Target: http://localhost:8000


In [2]:
try:
    status, health = get_json("/health")
    print(f"GET /health -> {status}")
    print(json.dumps(health, indent=2))
except urllib.error.HTTPError as e:
    print(f"GET /health -> HTTP {e.code}")
    print(e.read().decode("utf-8", errors="replace"))
except urllib.error.URLError as e:
    print("Connection failed — is the frontend listening on", BASE_URL, "?")
    raise

GET /health -> 200
{
  "status": "healthy",
  "endpoints": [
    "dyn://dynamo-system-vllm-disagg-015f4355.backend.clear_kv_blocks",
    "dyn://dynamo-system-vllm-disagg-015f4355.backend.generate",
    "dyn://dynamo-system-vllm-disagg-015f4355.prefill.clear_kv_blocks",
    "dyn://dynamo-system-vllm-disagg-015f4355.prefill.generate",
    "dyn://dynamo-system-vllm-disagg-015f4355.prefill.worker_kv_indexer_query_dp0"
  ],
  "instances": [
    {
      "component": "backend",
      "endpoint": "clear_kv_blocks",
      "namespace": "dynamo-system-vllm-disagg-015f4355",
      "instance_id": 4558425047763190,
      "transport": {
        "tcp": "10.244.4.187:45459/1031dd09b508f6/clear_kv_blocks"
      }
    },
    {
      "component": "backend",
      "endpoint": "generate",
      "namespace": "dynamo-system-vllm-disagg-015f4355",
      "instance_id": 4558425047763190,
      "transport": {
        "tcp": "10.244.4.187:45459/1031dd09b508f6/generate"
      }
    },
    {
      "component": "pref

In [3]:
status, models_doc = get_json("/v1/models")
print(f"GET /v1/models -> {status}")
print(json.dumps(models_doc, indent=2)[:4000])

model_ids = [m["id"] for m in models_doc.get("data", []) if "id" in m]
if not model_ids:
    raise RuntimeError("No models in /v1/models — register a backend before chat tests.")
MODEL = model_ids[0]
print(f"\nUsing model: {MODEL!r}")

GET /v1/models -> 200
{
  "object": "list",
  "data": [
    {
      "id": "Qwen/Qwen3-0.6B",
      "object": "model",
      "created": 1776362118,
      "owned_by": "nvidia"
    }
  ]
}

Using model: 'Qwen/Qwen3-0.6B'


In [4]:
chat_payload = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": "You are a helpful assistant. Reply in one short sentence."},
        {"role": "user", "content": "Say hello and confirm the Dynamo endpoint is working."},
    ],
    "max_tokens": 128,
    "temperature": 0.2,
    "stream": False,
}

status, completion = post_json("/v1/chat/completions", chat_payload)
print(f"POST /v1/chat/completions -> {status}")
print(json.dumps(completion, indent=2))

choice0 = completion["choices"][0]
msg = choice0.get("message", {})
print("\nAssistant:", msg.get("content", choice0))

POST /v1/chat/completions -> 200
{
  "id": "chatcmpl-33a6977d-78a6-4901-b432-1db5eace844f",
  "choices": [
    {
      "index": 0,
      "message": {
        "content": "<think>\nOkay, the user wants me to say hello and confirm that the Dynamo endpoint is working. Let me think about how to phrase this in one short sentence.\n\nFirst, \"Say hello\" is straightforward. Then, confirming the endpoint works could involve checking something like a response or a status. Maybe mention that the endpoint is accessible. I should keep it concise but clear. Let me try: \"Hello! The Dynamo endpoint is accessible and working.\" That covers both parts. I don't need any extra details. It's friendly and confirms the endpoint status.\n</think>\n\nHello! The Dynamo endpoint is accessible and working.",
        "role": "assistant",
        "reasoning_content": null
      },
      "finish_reason": "stop"
    }
  ],
  "created": 1776362123,
  "model": "Qwen/Qwen3-0.6B",
  "object": "chat.completion",
  "usag

### Optional: OpenAI client

If you have `openai` installed, you can point the client at the same base URL (no API key required for typical local setups).

In [5]:
try:
    from openai import OpenAI
except ImportError:
    print("Skip: pip install openai")
else:
    client = OpenAI(base_url=BASE_URL.rstrip("/") + "/v1", api_key="not-needed")
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Reply with the single word: ok"}],
        max_tokens=16,
    )
    print(r.choices[0].message.content)

self._post: <bound method SyncAPIClient.post of <openai.OpenAI object at 0x104b6a240>>
Extra headers: None
Request options: {}
>>>> REQUEST URL: /chat/completions
Sending HTTP Request: %s %s POST http://localhost:8000/v1/chat/completions
<think>
Okay, the user wants me to reply with the single word "ok
